# Suite JSN — JSON

JSON is a thin, strict (RFC 8259) encoding of Plain data, written exactly as Python's `json.dumps` writes it. These
cases cover strictness on output and input, text forms and encodings, value fidelity, and agreement with Plain.

In [ ]:
import { JSON, Plain, Proxies, Schemas as S } from "../src/Framework/index.js";
import { ValueError } from "../src/Framework/Errors.js";
import type { PlainMap } from "../src/Framework/Plain.js";
import { assert, equal, map, raises, same_graph, text } from "./support.js";

let Item = new S.OfObject.Builder().properties(text("s"), text("i", BigInt), text("f", Number), text("b", Boolean),
  text("raw", Uint8Array)).create();
const Link = new S.OfRelation.Builder().links("from", "to").properties(text("w", Number)).create();
Item = new S.OfObject.Builder(Item).relations((r) => r.name("out").of(Link).me("from"), (r) => r.name("in").of(Link).me("to")).update();
Proxies.register("Item", Item);
Proxies.register("Link", Link);
const B = Proxies.Builders;
const FromJSON = JSON.FromJSON(B);
const field = (textOut: string, name: string) =>
  (((JSON.loads(textOut) as PlainMap).get("objects") as PlainMap).get("s0") as PlainMap).get(name);

## JSN-01 · Output never contains NaN or Infinity literals

In [ ]:
for (const value of [NaN, Infinity, -Infinity]) {
  const out = JSON.ToJSON(Item, B.Item().f(value).create());
  assert(!/\b(NaN|Infinity)\b(?!")/.test(out.replaceAll('"NaN"', "").replaceAll('"Infinity"', "").replaceAll('"-Infinity"', "")));
  assert(["NaN", "Infinity", "-Infinity"].includes(field(out, "f") as string));
}
raises(ValueError, () => JSON.dumps(map({ x: NaN }))); // plain data should never hold one, and dumps refuses it

## JSN-02 · Input: non-standard constants, duplicate keys and syntax errors are rejected

In [ ]:
for (const bad of ['{"a": NaN}', '{"a": Infinity}', '{"a": -Infinity}', "[NaN]",
  '{"a": 1, "a": 2}', '{"x": {"a": 1, "a": 1}}', '[{"k": 1, "k": 1}]',
  '{"a": 1,}', "{'a': 1}", '{"a": 1} // c', '{"a": 01}', "", "   ", '{"a": 1} {"b": 2}', '{"a": .5}']) {
  raises(ValueError, () => JSON.loads(bad));
}

## JSN-03 · Text forms: str, UTF-8/16/32 bytes, and byte-order marks

In [ ]:
function encode(value: string, encoding: string): Uint8Array {
  const units = [...value].flatMap((c) => {
    const p = c.codePointAt(0) as number;
    return p > 0xffff ? [0xd800 + ((p - 0x10000) >> 10), 0xdc00 + ((p - 0x10000) & 0x3ff)] : [p];
  });
  const points = [...value].map((c) => c.codePointAt(0) as number);
  const u16 = (le: boolean) => units.flatMap((u) => (le ? [u & 0xff, u >> 8] : [u >> 8, u & 0xff]));
  const u32 = (le: boolean) => points.flatMap((p) => { const b = [p >>> 24, (p >> 16) & 0xff, (p >> 8) & 0xff, p & 0xff]; return le ? b.reverse() : b; });
  switch (encoding) {
    case "utf-16": return new Uint8Array([0xff, 0xfe, ...u16(true)]);
    case "utf-16-le": return new Uint8Array(u16(true));
    case "utf-16-be": return new Uint8Array(u16(false));
    case "utf-32": return new Uint8Array([0xff, 0xfe, 0x00, 0x00, ...u32(true)]);
    case "utf-8-sig": return new Uint8Array([0xef, 0xbb, 0xbf, ...new TextEncoder().encode(value)]);
    default: return new TextEncoder().encode(value);
  }
}
{
  const doc = '{"a": "é"}';
  assert(equal(JSON.loads(doc), map({ a: "é" })) && equal(JSON.loads(encode(doc, "utf-8")), map({ a: "é" })));
  for (const encoding of ["utf-16", "utf-16-le", "utf-16-be", "utf-32", "utf-8-sig"]) {
    assert(equal(JSON.loads(encode(doc, encoding)), map({ a: "é" })), encoding);
  }
  raises(ValueError, () => JSON.loads("﻿" + doc), "BOM"); // a BOM inside a str is an error; in bytes it is accepted
  raises(ValueError, () => JSON.loads(new Uint8Array([0xff, 0xfe, 0x00]))); // undecodable bytes
}

## JSN-04 · Output format: key order kept, non-ASCII kept, indentation optional

In [ ]:
{
  const item = B.Item().raw(new Uint8Array([1])).s("日本").i(1n).create();
  const compact = JSON.ToJSON(Item, item);
  assert(compact === '{"root": "s0", "objects": {"s0": {"s": "日本", "i": 1, "raw": "AQ=="}}}', compact);
  const pretty = JSON.ToJSON(Item, item, { indent: 2 });
  assert(pretty.includes("\n") && equal(JSON.loads(pretty), JSON.loads(compact)));
  assert(JSON.ToJSON.OfObject(Item, item, { indent: 4 }) === JSON.dumps(Plain.ToPlain(Item, item), { indent: 4 }));
}

## JSN-05 · Value fidelity: integers of any size, exact floats, every string

In [ ]:
for (const value of [0n, -1n, 2n ** 53n + 1n, 2n ** 100n, -(2n ** 200n)]) {
  assert((FromJSON(Item, JSON.ToJSON(Item, B.Item().i(value).create())) as any).i === value);
}
for (const value of [0.1, 1 / 3, 1e308, 5e-324, 2.2250738585072014e-308, -0.0, 123456789.123456789]) {
  const back = (FromJSON(Item, JSON.ToJSON(Item, B.Item().f(value).create())) as any).f;
  assert(Object.is(back, value));
}
for (const value of ["", "\x00", "\x1f\x7f", '"\\/', "  ", "\ud800", "\u{1f600}", "a\r\nb"]) {
  assert((FromJSON(Item, JSON.ToJSON(Item, B.Item().s(value).create())) as any).s === value, value);
}
{
  const all = new Uint8Array(Array.from({ length: 256 }, (_, i) => i));
  assert(equal((FromJSON(Item, JSON.ToJSON(Item, B.Item().raw(all).create())) as any).raw, all));
}

## JSN-06 · A lone surrogate is valid JSON text in memory but not encodable as UTF-8 (pinned)
Python raises when encoding it; JavaScript's `TextEncoder` silently replaces it with U+FFFD. Either way, callers
writing files should reject such strings.

In [ ]:
{
  const lone = JSON.ToJSON(Item, B.Item().s("\ud800").create());
  assert(lone.includes("\ud800"));
  const encoded = new TextDecoder().decode(new TextEncoder().encode(lone));
  assert(!encoded.includes("\ud800") && encoded.includes("�")); // lossy, where Python raises UnicodeEncodeError
}

## JSN-07 · Natives at the top level

In [ ]:
assert(JSON.ToJSON(new S.OfNative.Data(BigInt), 5n) === "5");
assert(JSON.ToJSON(new S.OfNative.Data(Uint8Array), new Uint8Array([0xff])) === '"/w=="');
assert(JSON.ToJSON.OfNative(new S.OfNative.Data(Number), Infinity) === '"Infinity"');
assert(FromJSON(new S.OfNative.Data(String), '"x"') === "x" && FromJSON.OfNative(new S.OfNative.Data(String), '"x"') === "x");
assert(JSON.ToJSON(new S.OfNative.Data(Number), 1.0) === "1.0"); // a float stays visibly a float
raises(TypeError, () => FromJSON(new S.OfNative.Data(BigInt), "true"));
raises(TypeError, () => FromJSON(new S.OfNative.Data(Number), "1")); // JSON 1 is an integer, not a float

## JSN-08 · JSON agrees with Plain, and graphs round trip

In [ ]:
{
  const hub = B.Item().s("hub").out((x: any) => x.to((y: any) => y.s("leaf1")).w(1.0))
    .out((x: any) => x.to((y: any) => y.s("leaf2")).w(NaN)).create();
  const graph = Plain.ToPlain.Reachable(Item, hub);
  const text_ = JSON.ToJSON.Reachable(Item, hub);
  assert(equal(JSON.loads(text_), graph));
  const back = FromJSON.Reachable(Item, text_) as Proxies.Instance;
  assert(back !== hub && same_graph(Plain.ToPlain.Reachable(Item, back), graph));
  raises(ValueError, () => FromJSON(Item, JSON.ToJSON(Item, hub)), "unresolved reference");
  raises(ValueError, () => FromJSON.Reachable(Item, text_.replace('"s1"', '"s9"')));
}